In [3]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://mongodb:27017/TiendaBigData.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 128 registros.
+--------------------+-------------------+--------+--------------------+-----+
|                 _id|      fecha_captura|   grupo|       identificador|valor|
+--------------------+-------------------+--------+--------------------+-----+
|69df95b6899879e86...|2026-04-15 13:41:34|Vannessa|18.5" Laptop, 24G...|489.0|
|69df95b6899879e86...|2026-04-15 13:41:34|Vannessa|Ryzen 5 7430U 15....|489.0|
|69df95b6899879e86...|2026-04-15 13:41:34|Vannessa|Laptop 15.6 Inch ...|349.0|
|69df95b6899879e86...|2026-04-15 13:41:34|Vannessa|15.6 Inch Laptop,...|289.0|
|69df95b6899879e86...|2026-04-15 13:41:34|Vannessa|Lenovo IdeaPad Sl...|449.0|
+--------------------+-------------------+--------+--------------------+-----+
only showing top 5 rows



In [4]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+------------------+
|    marca|   precio_promedio|
+---------+------------------+
|      MSI|            2258.0|
|       LG|            1305.0|
|     Acer|             962.0|
|     DELL| 955.3333333333334|
|     ASUS|             852.8|
|Microsoft|             825.0|
|   Lenovo| 785.0869565217391|
|     Dell|             725.0|
|    Ryzen|             544.0|
| ACEMAGIC|504.29411764705884|
|  Lenovo,|             492.5|
|    18.5"|             489.0|
|     acer|             462.5|
|        2| 412.3333333333333|
|       HP|             412.0|
|      GMR|             402.0|
|   Laptop|          366.5625|
|    15.6"|             349.0|
|     15.6| 327.7142857142857|
|   KEFEYA|             309.0|
+---------+------------------+
only showing top 20 rows

